In [ ]:
import os
import numpy as np
from PIL import Image
import warnings

warnings.filterwarnings("ignore")
from tqdm import tqdm
from pathlib import Path

BASE_DIR = Path("/home/y.osnovskaya/data/massachusetts_roads/raw_data")
OUTPUT_FILE = Path("/home/y.osnovskaya/data/massachusetts_roads/roads_patches_384.npy")
PATCH_SIZE = 384
STRIDE = 384

MAX_WHITE_RATIO = 0.70
BRIGHTNESS_THRESHOLD = 240

IMG_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMG_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def normalize(img):
    img = img.astype(np.float32) / 255.0
    return (img - IMG_MEAN) / IMG_STD


def find_mask(img_path, mask_dir):
    # Исправлено: ищем маску по имени, пробуя разные расширения
    stem = img_path.stem
    for ext in [".tiff", ".tif", ".png", ".jpg"]:
        mask_path = mask_dir / (stem + ext)
        if mask_path.exists():
            return mask_path
    return None


def is_valid_patch(img_patch):
    white_pixels = np.all(img_patch > BRIGHTNESS_THRESHOLD, axis=-1)
    white_ratio = white_pixels.sum() / white_pixels.size
    return white_ratio <= MAX_WHITE_RATIO


def extract_patches(img_arr, mask_arr):
    H, W = img_arr.shape[:2]
    valid_patches = []
    kept = 0
    skipped = 0

    if H < PATCH_SIZE or W < PATCH_SIZE:
        return valid_patches, kept, skipped

    for y in range(0, H - PATCH_SIZE + 1, STRIDE):
        for x in range(0, W - PATCH_SIZE + 1, STRIDE):
            p_img = img_arr[y : y + PATCH_SIZE, x : x + PATCH_SIZE]
            p_mask = mask_arr[y : y + PATCH_SIZE, x : x + PATCH_SIZE]

            if not is_valid_patch(p_img):
                skipped += 1
                continue

            p_img_norm = normalize(p_img)
            p_mask_bin = (p_mask > 127).astype(np.uint8)
            patch = np.concatenate([p_img_norm, p_mask_bin[..., None]], axis=-1)
            valid_patches.append(patch)
            kept += 1

    return valid_patches, kept, skipped


def main():
    all_patches = []
    total_kept = 0
    total_skipped = 0
    splits = ["train", "val", "test"]
    debug_mode = True

    for split in splits:
        img_dir = BASE_DIR / "tiff" / split
        mask_dir = BASE_DIR / "tiff" / f"{split}_labels"

        if not img_dir.exists() or not mask_dir.exists():
            print(f"Папки для {split} не найдены")
            continue

        img_files = sorted(list(img_dir.glob("*.tiff")))
        if not img_files:
            img_files = sorted(list(img_dir.glob("*.tif")))
        if not img_files:
            print(f" В {split} нет изображений")
            continue

        print(f"\n {split}: {len(img_files)} изображений")

        for idx, img_path in enumerate(tqdm(img_files, desc=f"Нарезка {split}")):
            if debug_mode and idx < 3:
                print(f"\n[DEBUG] Файл: {img_path.name}")
                mask_path = find_mask(img_path, mask_dir)
                print(
                    f"  Маска найдена: {mask_path is not None} ({mask_path.name if mask_path else 'Нет'})"
                )
                if mask_path is None:
                    samples = [f.name for f in mask_dir.iterdir()][:3]
                    print(f"  Реальные файлы в папке масок: {samples}")
                debug_mode = False
            else:
                mask_path = find_mask(img_path, mask_dir)

            if mask_path is None:
                continue

            try:
                img = np.array(Image.open(img_path).convert("RGB"))
                mask = np.array(Image.open(mask_path).convert("L"))
            except Exception as e:
                if idx < 5:
                    print(f"Ошибка чтения {img_path.name}: {e}")
                continue

            patches, kept, skipped = extract_patches(img, mask)
            all_patches.extend(patches)
            total_kept += kept
            total_skipped += skipped

    total_processed = total_kept + total_skipped
    print("\n" + "=" * 60)
    if total_processed == 0:
        print("ПАТЧИ НЕ СОЗДАНЫ. Проверьте вывод [DEBUG] выше.")
        return

    print(f"ФИЛЬТРАЦИЯ (Белый шум >{int(MAX_WHITE_RATIO*100)}%):")
    print("=" * 60)
    print(f"Принято: {total_kept:>6,} ({total_kept/total_processed:.1%})")
    print(f"Отброшено: {total_skipped:>6,} ({total_skipped/total_processed:.1%})")
    print("=" * 60)

    dataset = np.stack(all_patches, axis=0)
    print(f"\nУСПЕХ! Форма: {dataset.shape}, Размер: {dataset.nbytes/1e9:.2f} ГБ")
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(OUTPUT_FILE, dataset)
    print(f"Сохранено: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()